# 03 — Embedding Models: So Sánh Chất Lượng Embedding

**Vai trò:** Model Engineer · **Task:** S4-ME-02 (Yêu cầu 9.4)

`config/models.yaml` liệt kê hai embedding model được hỗ trợ: `nomic-embed-text` (768 chiều, mặc định) và `mxbai-embed-large` (1024 chiều). Notebook này so sánh chúng trên ba khía cạnh người học cần biết khi chọn embedding model cho RAG: **số chiều**, **tính tất định** (Property 3/4 — Yêu cầu 3.1, 3.2), và **chất lượng phân biệt ngữ nghĩa** (đoạn liên quan có điểm tương đồng cao hơn đoạn không liên quan hay không).

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.generation.llm_client import OllamaClient
from src.pipeline.experiment_tracker import ExperimentTracker

print(f"Project root: {PROJECT_ROOT}")

## 1. Phát hiện embedding model nào đã pull về OLLAMA

Cũng như notebook `02_model_comparison`, ta lọc ra trong số các model ứng viên (`config/models.yaml`) những model **thực sự có sẵn**, để notebook luôn chạy hết mà không phát sinh exception dù máy chưa pull đủ cả hai (DoD Sprint 4 — "không có exception chưa xử lý dù thiếu model").

In [ ]:
probe_client = OllamaClient()
pulled_models = probe_client.list_models()

CANDIDATE_EMBEDDING_MODELS = ["nomic-embed-text", "mxbai-embed-large"]
usable_embedding_models = [
    name for name in CANDIDATE_EMBEDDING_MODELS
    if any(pulled.split(":")[0] == name for pulled in pulled_models)
]

print(f"Model da pull             : {pulled_models or '(khong co)'}")
print(f"Embedding model co the so sanh: {usable_embedding_models or '(chua co model nao duoc pull)'}")

if not usable_embedding_models:
    print(
        "\n⚠️  Chua co embedding model ung vien nao duoc pull. Hay chay:\n"
        "    ollama pull nomic-embed-text\n"
        "    ollama pull mxbai-embed-large\n"
        "roi chay lai notebook de so sanh thuc."
    )

## 2. Số chiều, tính tất định, và thời gian nhúng

Với mỗi embedding model khả dụng: nhúng cùng một câu hai lần để kiểm chứng **tính tất định** (`embed_text(x) == embed_text(x)` — Property 3, Yêu cầu 3.1) và đo thời gian `embed_batch()` xử lý một lô câu mẫu.

In [ ]:
SAMPLE_TEXTS = [
    "RAG ket hop retrieval va generation de tra loi cau hoi dua tren tai lieu rieng.",
    "ChromaDB la mot vector database dung de luu tru va tim kiem embedding.",
    "Pho la mon an truyen thong noi tieng cua Viet Nam voi nuoc dung dam da.",
]

embedding_profiles = {}
for model_name in usable_embedding_models:
    model = OllamaEmbeddingModel(model_name=model_name)

    start = time.perf_counter()
    vectors = model.embed_batch(SAMPLE_TEXTS)
    batch_latency_ms = (time.perf_counter() - start) * 1000

    repeat_vector = model.embed_text(SAMPLE_TEXTS[0])
    is_deterministic = np.allclose(vectors[0], repeat_vector)

    embedding_profiles[model_name] = {
        "model": model,
        "vectors": np.array(vectors),
        "dimension": model.dimension,
        "batch_latency_ms": batch_latency_ms,
        "deterministic": is_deterministic,
    }
    print(
        f"[{model_name}] chieu={model.dimension}  "
        f"tat_dinh={is_deterministic}  "
        f"embed_batch({len(SAMPLE_TEXTS)} cau)={batch_latency_ms:.1f} ms"
    )
    assert is_deterministic, f"{model_name}: embed_text phai tat dinh (Property 3)"
    assert vectors[0] == repeat_vector or np.allclose(vectors[0], repeat_vector)

## 3. Chất lượng phân biệt ngữ nghĩa — similarity câu liên quan vs không liên quan

Một embedding "tốt" cho RAG phải đặt **câu hỏi và đoạn tài liệu liên quan** gần nhau hơn (cosine similarity cao hơn) so với một đoạn **không liên quan**. Đây chính là tính chất mà `ChromaVectorStore.similarity_search()` (Property 6/7/8) dựa vào để truy xuất đúng ngữ cảnh — notebook đo trực tiếp khoảng cách đó cho từng embedding model.

In [ ]:
def cosine_similarity(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


QUERY = "Vector database luu embedding nhu the nao?"
RELEVANT_TEXT = SAMPLE_TEXTS[1]      # ve ChromaDB / vector database
IRRELEVANT_TEXT = SAMPLE_TEXTS[2]    # ve mon an pho — khong lien quan

quality_rows = []
for model_name, profile in embedding_profiles.items():
    model = profile["model"]
    query_vec = model.embed_text(QUERY)
    relevant_vec = model.embed_text(RELEVANT_TEXT)
    irrelevant_vec = model.embed_text(IRRELEVANT_TEXT)

    sim_relevant = cosine_similarity(query_vec, relevant_vec)
    sim_irrelevant = cosine_similarity(query_vec, irrelevant_vec)
    quality_rows.append({
        "model": model_name,
        "dimension": profile["dimension"],
        "sim_lien_quan": round(sim_relevant, 4),
        "sim_khong_lien_quan": round(sim_irrelevant, 4),
        "khoang_cach_phan_biet": round(sim_relevant - sim_irrelevant, 4),
        "phan_biet_dung": sim_relevant > sim_irrelevant,
    })
    print(
        f"[{model_name}] sim(lien_quan)={sim_relevant:.4f}  "
        f"sim(khong_lien_quan)={sim_irrelevant:.4f}  "
        f"=> phan biet dung: {sim_relevant > sim_irrelevant}"
    )

df_quality = pd.DataFrame(quality_rows)

## 4. Bảng tổng hợp & ghi lại thực nghiệm

Gộp số chiều, latency và khả năng phân biệt ngữ nghĩa vào một bảng so sánh duy nhất, rồi log lại qua `ExperimentTracker` (Yêu cầu 9.8) để có thể đối chiếu giữa các lần thử (`compare_sessions()`).

In [ ]:
if embedding_profiles:
    df_summary = df_quality.copy()
    df_summary["batch_latency_ms"] = [
        round(embedding_profiles[m]["batch_latency_ms"], 1) for m in df_summary["model"]
    ]
    display(df_summary[[
        "model", "dimension", "batch_latency_ms",
        "sim_lien_quan", "sim_khong_lien_quan", "khoang_cach_phan_biet", "phan_biet_dung",
    ]])

    tracker = ExperimentTracker()
    for _, row in df_summary.iterrows():
        tracker.log_indexing(
            doc_id=f"embedding_models_probe::{row['model']}",
            chunk_strategy="n/a",
            chunk_size=int(row["dimension"]),
            num_chunks=len(SAMPLE_TEXTS),
            latency_ms=embedding_profiles[row["model"]]["batch_latency_ms"],
        )
    print("\nTom tat phien thuc nghiem:")
    print(tracker.get_summary())
else:
    print("Chua co embedding model nao de tong hop — xem huong dan o muc 1.")

## 5. Tổng kết

- Số chiều (`dimension`) khác nhau giữa các embedding model (`nomic-embed-text` = 768, `mxbai-embed-large` = 1024) phản ánh dung lượng biểu diễn ngữ nghĩa khác nhau — chiều cao hơn không phải lúc nào cũng "tốt hơn", mà đánh đổi với tốc độ và dung lượng lưu trữ trong `ChromaVectorStore`.
- Cả hai model đều phải **tất định** (Property 3/4 — Yêu cầu 3.1, 3.2): notebook xác minh trực tiếp `embed_text(x) == embed_text(x)`.
- Phép đo "khoảng cách phân biệt" (`sim_lien_quan - sim_khong_lien_quan`) là một proxy đơn giản cho chất lượng embedding trong RAG — giá trị càng dương, model càng tách bạch tốt giữa ngữ cảnh liên quan và không liên quan, giúp `similarity_search()` truy xuất chính xác hơn.
- Kết quả được ghi qua `ExperimentTracker` mà không làm gián đoạn notebook (Yêu cầu 9.8), sẵn sàng để so sánh giữa các phiên thực nghiệm khác nhau.